In [35]:
import os
import io
import ast
import fitz
import json
import torch
import base64
import requests
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv

from qdrant_client import QdrantClient, models
from qdrant_client.models import QueryRequest, Prefetch, Fusion

from colpali_engine.models import ColQwen2, ColQwen2Processor

from fastembed import SparseTextEmbedding

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

In [36]:
splade_model = SparseTextEmbedding(model_name="prithivida/Splade_PP_en_v1")

Fetching 5 files: 100%|██████████| 5/5 [00:30<00:00,  6.02s/it]


In [3]:
import torch

print("=== CUDA GPU STATUS CHECK ===")
cuda_available = torch.cuda.is_available()
print(f"Is CUDA available in this notebook?: {cuda_available}")

if cuda_available:
    print(f"Active Graphics Card Name: {torch.cuda.get_device_name(0)}")
else:
    print("PyTorch cannot find your GPU.")


=== CUDA GPU STATUS CHECK ===
Is CUDA available in this notebook?: True
Active Graphics Card Name: NVIDIA RTX 5000 Ada Generation Laptop GPU


In [9]:
load_dotenv()

True

In [5]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [23]:
client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)

In [10]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

Loading weights: 100%|██████████| 392/392 [00:00<00:00, 2005.95it/s]
ColQwen2 LOAD REPORT from: vidore/colqwen2-v1.0
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
base_model.language_model.model.custom_text_proj.lora_B.default.weight | UNEXPECTED | 
base_model.language_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED | 
custom_text_proj.lora_A.default.weight                                 | MISSING    | 
custom_text_proj.lora_B.default.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [14]:
def get_coarse_embedding(embeddings_tensor):
    pooled = embeddings_tensor.mean(dim=1) # mean pooling 
    normalised = pooled / pooled.norm(dim=-1, keepdim=True) # L2 normalization 
    return normalised.squeeze().to(torch.float32).cpu()

In [27]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True) # L2 normalization

        coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu().tolist()
    return coarse_vector, embeddings

In [28]:
query = "What is 1 + 1"

In [37]:
coarse_vector, page_embeddings = embed_query(query, model, processor)
splade_vector = splade_model.query_embed(query)

In [ ]:
def search(coarse_vector, page_embeddings):
    response = client.query_points(
        collection_name="pages",

        prefetch=[
            models.Prefetch(
                query=models.Fusion.RRF,
                prefetch=[
                    models.Prefetch(query=coarse_vector, using="coarse_vector", limit=100),
                    models.Prefetch(query=splade_vector, using="splade_vector", limit=100),
                ],
                limit=50
            )
        ],
        query=,
        using=""     
    )